In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
WORK_DIR="/home/magnolia/DataScience/Stellar_Class"
TRAIN_PATH=Path(WORK_DIR,"data/train.csv")
TEST_PATH=Path(WORK_DIR, "data/test.csv")
MODEL_DIR=Path(WORK_DIR,"models")

df=pd.read_csv(TRAIN_PATH).set_index('id')
target=df['class']
X=df.drop(['class'], axis=1)
# map targets to numericals 

target_map={"GALAXY":0, "QSO": 1, "STAR": 2}
rev_map={0:"GALAXY", 1:"QSO", 2:"STAR"}

target=target.map(target_map)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, target, stratify=target, random_state=10)

In [ ]:
num_cols=X.select_dtypes(['float', 'int']).columns.to_list()
cat_cols=X.select_dtypes('object').columns.to_list()
column_est=ColumnTransformer([('num_est', MinMaxScaler(), num_cols),
                              ('cat_est', OneHotEncoder(), cat_cols)])
pipeline=Pipeline([('column_transformer', column_est)])

In [ ]:
X_train=pipeline.fit_transform(X_train)
X_test=pipeline.transform(X_test)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score

model=HistGradientBoostingClassifier(learning_rate=0.1, 
                                     l2_regularization=0.5, 
                                     max_features=0.82,
                                     max_depth=12)
model.fit(X_train, y_train)
print(f"Train: {balanced_accuracy_score(y_train, model.predict(X_train))}\nTest: {balanced_accuracy_score(y_test, model.predict(X_test))}")

In [ ]:
# ^^^^ Optimise Hyper Parameters and Update the above model then predict and submit. ^^^^

In [ ]:
# Learning Rate Optimisation
learning_rate_ = np.linspace(0.01,0.1,20)
train_         = []
test_          = []
for rate in learning_rate_:
    model       = HistGradientBoostingClassifier(learning_rate=rate, random_state=10)
    model.fit(X_train, y_train)
    train_score = balanced_accuracy_score(y_train, model.predict(X_train))
    test_score  = balanced_accuracy_score(y_test, model.predict(X_test))
    train_.append(train_score), test_.append(test_score)
    print(f"Learning Rate: {rate}\nTrain: {train_score}\nTest: {test_score}\n")
    
hgb_learning_rate_df=pd.DataFrame({'learning_rate': learning_rate_, 'train': train_, 'test_': test_})
plt.plot(hgb_learning_rate_df['learning_rate'], hgb_learning_rate_df['train'], label='Train')
plt.plot(hgb_learning_rate_df['learning_rate'],hgb_learning_rate_df['test_'], label='Test')
plt.legend()

In [ ]:
# L2 Optimisation 
l2_penalty_ = np.linspace(0,1,11)
train_     = []
test_      = []
for penalty in l2_penalty_:
    model=HistGradientBoostingClassifier(learning_rate=0.1, l2_regularization=penalty)
    model.fit(X_train, y_train)
    train_score = balanced_accuracy_score(y_train, model.predict(X_train))
    test_score  = balanced_accuracy_score(y_test, model.predict(X_test))
    train_.append(train_score), test_.append(test_score)
    print(f"L2: {penalty}\nTrain: {train_score}\nTest: {test_score}\n")

hgb_l2_reg_df=pd.DataFrame({'l2': l2_penalty_, 'train': train_, 'test': test_})
plt.plot(hgb_l2_reg_df['l2'], hgb_l2_reg_df['train'], label='Train')
plt.plot(hgb_l2_reg_df['l2'], hgb_l2_reg_df['test'], label='Test')
plt.legend()

In [ ]:
# Max Features Optimisation
max_feat_   = np.linspace(0.1,1,11)
train_      = []
test_       = []

for feat in max_feat_:
    model=HistGradientBoostingClassifier(learning_rate=0.1, l2_regularization=0.5, max_features=feat)
    model.fit(X_train, y_train)
    train_score = balanced_accuracy_score(y_train, model.predict(X_train))
    test_score  = balanced_accuracy_score(y_test, model.predict(X_test))
    train_.append(train_score), test_.append(test_score)
    print(f"Max_features: {feat}\nTrain: {train_score}\nTest: {test_score}\n")

hgb_max_feat=pd.DataFrame({'max_feat': max_feat_, 'train': train_, 'test': test_})
plt.plot(hgb_max_feat['max_feat'], hgb_max_feat['train'], label='Train')
plt.plot(hgb_max_feat['max_feat'], hgb_max_feat['test'], label='Test')
plt.legend()

In [ ]:
# Max Depth Optimisation
max_depth_  = np.arange(1,20)
train_      = []
test_       = []

for depth in max_depth_:
    model=HistGradientBoostingClassifier(max_depth=depth)
    model.fit(X_train, y_train)
    train_score = balanced_accuracy_score(y_train, model.predict(X_train))
    test_score  = balanced_accuracy_score(y_test, model.predict(X_test))
    train_.append(train_score), test_.append(test_score)
    print(f"max_depth: {depth}\nTrain: {train_score}\nTest: {test_score}\n")

In [ ]:
hgb_max_depth=pd.DataFrame({'depth':max_depth_, 'train': train_, 'test': test_})
plt.plot(hgb_max_depth['depth'], hgb_max_depth['train'], label='Train')
plt.plot(hgb_max_depth['depth'], hgb_max_depth['test'], label='Test')

In [ ]:
# Prediction Below.

In [ ]:
# Prediction
df_pred=pd.read_csv(TEST_PATH).set_index('id')
df_pred_transform=pipeline.transform(df_pred)
y_pred=model.predict(df_pred_transform)
y_pred=[rev_map[p] for p in y_pred]
pd.DataFrame({'id':df_pred.index, 'class':y_pred}).set_index('id').to_csv(Path(WORK_DIR, 'submission', 'submission6.csv'))

In [ ]:
# HORRIBLE PERFORMANCE BECAUSE MY TRAINING SET IS LEAKING INTO MY TEST SET OMG!!!!!!!!!!!!!!!!!!!!!!!
# I was applying the scale transformations all wrong. ;;;-;;;